In [3]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

# Download VADER if not already
nltk.download('vader_lexicon')

# -----------------------
# PATHS
# -----------------------
TRANSCRIPT_FOLDER = "cleaned_transcripts"
MERGED_LABELS_PATH = "merged_labels.csv"
OUTPUT_CSV = "text_features2.csv"

# -----------------------
# LOAD MODELS
# -----------------------
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim
vader = SentimentIntensityAnalyzer()

# -----------------------
# WORD LISTS
# -----------------------
pronouns = set([
    "i","me","my","mine","myself","we","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself","she","her","hers","herself",
    "it","its","itself","they","them","their","theirs","themselves"
])

negations = set(["not","no","never","n't"])
uncertainty = set(["maybe","perhaps","probably","possibly","might","could"])

# -----------------------
# LOAD LABELS
# -----------------------
labels_df = pd.read_csv(MERGED_LABELS_PATH)
allowed_ids = set(labels_df["Participant_ID"].astype(str))

# -----------------------
# FEATURE FUNCTION
# -----------------------
def extract_text_features(text):

    words = text.split()
    sentences = text.split(".")

    # Basic counts
    word_count = len(words)
    sentence_count = len(df)

    avg_sentence_length = word_count / (sentence_count + 1e-6)

    # Ratios
    pronoun_count = sum(1 for w in words if w.lower() in pronouns)
    pronoun_ratio = pronoun_count / (word_count + 1e-6)

    negation_count = sum(1 for w in words if w.lower() in negations)
    uncertainty_count = sum(1 for w in words if w.lower() in uncertainty)

    # Type-token ratio
    unique_words = len(set(words))
    type_token_ratio = unique_words / (word_count + 1e-6)

    # -----------------------
    # VADER Sentiment
    # -----------------------
    sentiment = vader.polarity_scores(text)

    # -----------------------
    # SBERT
    # -----------------------
    embedding = sbert_model.encode(text)

    features = {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_length": avg_sentence_length,
        "pronoun_ratio": pronoun_ratio,
        "negation_count": negation_count,
        "uncertainty_count": uncertainty_count,
        "type_token_ratio": type_token_ratio,

        "sentiment_positive": sentiment["pos"],
        "sentiment_negative": sentiment["neg"],
        "sentiment_neutral": sentiment["neu"],
        "sentiment_compound": sentiment["compound"]
    }

    # Add SBERT features
    for i, val in enumerate(embedding):
        features[f"sbert_{i}"] = val

    return features

# -----------------------
# MAIN LOOP
# -----------------------
all_data = []

for file in tqdm(os.listdir(TRANSCRIPT_FOLDER)):

    if not file.endswith(".csv"):
        continue

    participant_id = file.split("_")[0]

    # Filter using merged labels
    if participant_id not in allowed_ids:
        continue

    try:
        df = pd.read_csv(os.path.join(TRANSCRIPT_FOLDER, file))
        text = " ".join(df["value"].astype(str))

        features = extract_text_features(text)
        features["participant_id"] = participant_id

        # Add label
        label = labels_df[
            labels_df["Participant_ID"].astype(str) == participant_id
        ]["PHQ8_Binary"].values

        features["label"] = label[0] if len(label) > 0 else np.nan

        all_data.append(features)

    except Exception as e:
        print("Error:", file, e)

# -----------------------
# SAVE CSV
# -----------------------
df = pd.DataFrame(all_data)

# Reorder columns
cols = df.columns.tolist()
cols.remove("participant_id")
cols.remove("label")

df = df[["participant_id"] + cols + ["label"]]

df.to_csv(OUTPUT_CSV, index=False)

print("✅ Saved:", OUTPUT_CSV)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
100%|██████████| 188/188 [00:08<00:00, 22.09it/s]


✅ Saved: text_features2.csv


UPDATED TEXT FEATURE EXTRACTION

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
import nltk

# -----------------------
# PATHS
# -----------------------
TRANSCRIPT_FOLDER = "cleaned_transcripts"
MERGED_LABELS_PATH = "merged_labels.csv"
OUTPUT_CSV = "text_features_updated.csv"

# -----------------------
# LOAD MODEL
# -----------------------
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim

# -----------------------
# WORD LISTS
# -----------------------
first_person = set(["i","me","my","mine","myself"])
negations = set(["not","no","never","n't"])
uncertainty = set(["maybe","perhaps","probably","possibly","might","could"])

# -----------------------
# LOAD LABELS
# -----------------------
labels_df = pd.read_csv(MERGED_LABELS_PATH)
allowed_ids = set(labels_df["Participant_ID"].astype(str))

# -----------------------
# FEATURE FUNCTION
# -----------------------
def extract_text_features(df):

    sentences = df["value"].astype(str).tolist()

    # -----------------------
    # Remove short / useless sentences
    # -----------------------
    sentences = [s for s in sentences if len(s.split()) > 3]

    if len(sentences) == 0:
        return None

    full_text = " ".join(sentences)
    words = full_text.split()

    # -----------------------
    # BASIC FEATURES
    # -----------------------
    word_count = len(words)
    sentence_count = len(sentences)
    avg_sentence_length = word_count / (sentence_count + 1e-6)

    # -----------------------
    # LINGUISTIC FEATURES
    # -----------------------
    first_person_ratio = sum(1 for w in words if w.lower() in first_person) / (word_count + 1e-6)
    negation_count = sum(1 for w in words if w.lower() in negations)
    uncertainty_count = sum(1 for w in words if w.lower() in uncertainty)

    unique_words = len(set(words))
    type_token_ratio = unique_words / (word_count + 1e-6)

    # -----------------------
    # PAUSE FEATURES (VERY IMPORTANT)
    # -----------------------
    pauses = []

    for i in range(len(df)-1):
        try:
            gap = df.iloc[i+1]["start_time"] - df.iloc[i]["stop_time"]
            if gap > 0:
                pauses.append(gap)
        except:
            continue

    if len(pauses) > 0:
        mean_pause = np.mean(pauses)
        max_pause = np.max(pauses)
    else:
        mean_pause = 0
        max_pause = 0

    # -----------------------
    # SBERT (sentence-level)
    # -----------------------
    embeddings = sbert_model.encode(sentences)

    mean_emb = np.mean(embeddings, axis=0)
    std_emb = np.std(embeddings, axis=0)

    # -----------------------
    # FEATURE DICTIONARY
    # -----------------------
    features = {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_sentence_length": avg_sentence_length,
        "first_person_ratio": first_person_ratio,
        "negation_count": negation_count,
        "uncertainty_count": uncertainty_count,
        "type_token_ratio": type_token_ratio,
        "mean_pause": mean_pause,
        "max_pause": max_pause
    }

    # Add SBERT mean + std
    for i, val in enumerate(mean_emb):
        features[f"sbert_mean_{i}"] = val

    for i, val in enumerate(std_emb):
        features[f"sbert_std_{i}"] = val

    return features

# -----------------------
# MAIN LOOP
# -----------------------
all_data = []

for file in tqdm(os.listdir(TRANSCRIPT_FOLDER)):

    if not file.endswith(".csv"):
        continue

    participant_id = file.split("_")[0]

    if participant_id not in allowed_ids:
        continue

    try:
        df = pd.read_csv(os.path.join(TRANSCRIPT_FOLDER, file))

        features = extract_text_features(df)

        if features is None:
            continue

        features["participant_id"] = participant_id

        label = labels_df[
            labels_df["Participant_ID"].astype(str) == participant_id
        ]["PHQ8_Binary"].values

        features["label"] = label[0] if len(label) > 0 else np.nan

        all_data.append(features)

    except Exception as e:
        print("Error:", file, e)

# -----------------------
# SAVE CSV
# -----------------------
df = pd.DataFrame(all_data)

cols = df.columns.tolist()
cols.remove("participant_id")
cols.remove("label")

df = df[["participant_id"] + cols + ["label"]]

df.to_csv(OUTPUT_CSV, index=False)

print("✅ Saved:", OUTPUT_CSV)

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 188/188 [00:51<00:00,  3.64it/s]

✅ Saved: text_features_updated.csv
